# Qwen3-0.6B family-isolated QLoRA training template

**This notebook is a prepared runner, not a claim that the current corpus is ready.** It must stop if the uploaded train/dev files are not reviewed under the explicit AI single-reviewer contract or are not family-disjoint. Do not upload `blind.private.jsonl` or blind labels. Eval loss is only an optimization diagnostic; factual quality and new-family pass require the separate evaluator using explicit AI single-reviewer review labels.

Attach the reviewed train/dev corpus and this repository's `tools/micro-generator/` directory as Kaggle inputs. The current review queue is intentionally ineligible and will fail preflight.

In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
import torch
print('python', sys.version)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required; this notebook refuses CPU fine-tuning')
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout[:4000])
print('GPU memory GiB', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Keep the Kaggle-provided CUDA/PyTorch build; install only the model-training stack.
packages = ['transformers>=4.51,<5', 'peft>=0.15,<1', 'accelerate>=1.6,<2', 'bitsandbytes>=0.45,<1', 'huggingface_hub>=0.30,<1']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *packages])
print('Installed:', packages)

In [ ]:
# Adjust these Kaggle input paths after attaching the reviewed artifacts.
PIPELINE_CANDIDATES = [Path('/kaggle/input/cfb-micro-generator-pipeline/tools/micro-generator'), Path('/kaggle/input/cfb-micro-generator-pipeline')]
PIPELINE_DIR = next((candidate for candidate in PIPELINE_CANDIDATES if (candidate / 'train_qlora.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Attach the tools/micro-generator files as Kaggle input cfb-micro-generator-pipeline')
CORPUS_DIR = Path('/kaggle/input/cfb-micro-generator-corpus')
TRAIN_FILE = CORPUS_DIR / 'train.jsonl'
DEV_FILE = CORPUS_DIR / 'dev.jsonl'
TRAINER = PIPELINE_DIR / 'train_qlora.py'
for required in (TRAINER, PIPELINE_DIR / 'prompt-v1.txt', TRAIN_FILE, DEV_FILE):
    if not required.is_file():
        raise FileNotFoundError(f'Missing reviewed input: {required}')
MODEL_ID = 'Qwen/Qwen3-0.6B'
from huggingface_hub import model_info
MODEL_REVISION = model_info(MODEL_ID).sha
assert len(MODEL_REVISION) == 40, MODEL_REVISION
print('Pinned model commit:', MODEL_REVISION)
print('Data files:', TRAIN_FILE.stat().st_size, DEV_FILE.stat().st_size)

In [ ]:
# Fail-closed data/provenance preflight; it checks complete reviewed facts, target review, and family isolation.
subprocess.run([sys.executable, str(TRAINER), '--train', str(TRAIN_FILE), '--dev', str(DEV_FILE), '--preflight-only'], check=True)
print('Preflight passed. This only permits a development training run; it does not certify factual quality.')

In [ ]:
# Development run only. Do not expose blind examples, facts, references, or outputs to this cell.
OUT = Path('/kaggle/working/qwen3-0.6b-qlora-run')
subprocess.run([
    sys.executable, str(TRAINER), '--train', str(TRAIN_FILE), '--dev', str(DEV_FILE),
    '--model-id', MODEL_ID, '--revision', MODEL_REVISION, '--output', str(OUT),
    '--max-length', '8192', '--epochs', '3', '--batch-size', '1',
    '--gradient-accumulation', '8', '--learning-rate', '0.0002', '--lora-rank', '16'
], check=True)
print('Run finished:', OUT)

In [ ]:
# Dev predictions are emitted for AI single-reviewer review; they are not automatically scored as semantically correct.
PREDICTOR = PIPELINE_DIR / 'run_hf.py'
DEV_PREDICTIONS = OUT / 'dev-predictions.pending-review.jsonl'
subprocess.run([
    sys.executable, str(PREDICTOR), '--input', str(DEV_FILE), '--out', str(DEV_PREDICTIONS),
    '--base-model', MODEL_ID, '--revision', MODEL_REVISION, '--adapter', str(OUT / 'adapter-final'),
    '--split', 'dev', '--max-new-tokens', '512', '--max-input-tokens', '8192', '--four-bit'
], check=True)
print('AI single-reviewer review pending:', DEV_PREDICTIONS)

In [ ]:
# Bundle adapter, dev predictions and provenance for download. Blind data is never included.
archive = shutil.make_archive('/kaggle/working/cfb-micro-generator-run', 'zip', root_dir=OUT)
print('Download artifact:', archive, Path(archive).stat().st_size, 'bytes')
print('Dev predictions remain pending AI single-reviewer review; this archive is not a factual-quality pass.')